# Q3 — Steel Manufacturing LP: Environmental Policy Analysis (Parts b/c/d)

**Course:** Operations Research 1 — Khalifa University  

This notebook extends the base case model (`q3_part_a_base_case.ipynb`) by adding environmental constraints representing a policy requiring a **20% reduction** in water, electricity, fuel, or pollutant levels by 2035 relative to the 2025 baseline.

## 2025 Baseline Environmental Levels (from base case)
| Resource | 2025 Baseline | 80% Target (2035) |
|----------|--------------|-------------------|
| Water | 690 units | 552 units |
| Electricity | 480 units | 384 units |
| Fuel | 960 units | 768 units |
| Pollutant | 1,260 units | 1,008 units |

## How to Use This Notebook
In **Section 3**, uncomment exactly ONE environmental constraint block at a time and re-run to see its impact.  
For Part d, uncomment BOTH the water and pollutant constraints simultaneously.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

## 1. Data

In [ ]:
processes = [1, 2, 3]

cap0     = {1: 120, 2: 50,  3: 0}    # Installed capacity in 2025 (kt)
delta    = {1: 100, 2: 100, 3: 80}   # Maximum expansion by 2035 (kt)
op_cost  = {1: 300, 2: 400, 3: 200}  # Operational cost ($/kt)
inv_cost = {1: 3000, 2: 4000, 3: 5000}  # Investment cost for new capacity ($/kt/yr)

# Environmental use/emission rates per kt produced
water       = {1: 5,  2: 3, 3: 1}
electricity = {1: 3,  2: 4, 3: 10}
fuel        = {1: 6,  2: 8, 3: 3}
pollutant   = {1: 10, 2: 2, 3: 5}

demand2025 = 150  # kt
demand2035 = 200  # kt

# 2035 environmental targets = 80% of 2025 baseline levels
water_target       = 690  * 0.80   # 552 units
electricity_target = 480  * 0.80   # 384 units
fuel_target        = 960  * 0.80   # 768 units
pollutant_target   = 1260 * 0.80   # 1008 units

## 2. Build the Model

In [ ]:
m = gp.Model("Steel_Environmental")
m.setParam('OutputFlag', 1)

# Decision variables
x25 = m.addVars(processes, name="x2025", lb=0)
x35 = m.addVars(processes, name="x2035", lb=0)
E   = m.addVars(processes, name="Expand", lb=0)

# Objective: minimise total operational + investment cost
m.setObjective(
    gp.quicksum(op_cost[p] * x25[p] for p in processes)
    + gp.quicksum(op_cost[p] * x35[p] + inv_cost[p] * E[p] for p in processes),
    GRB.MINIMIZE
)

# Capacity and demand constraints (same as base case)
cap25_constrs = {}
for p in processes:
    cap25_constrs[p] = m.addConstr(x25[p] <= cap0[p],          name=f"Cap25_p{p}")
    m.addConstr(x35[p] <= cap0[p] + E[p],                      name=f"Cap35_p{p}")
    m.addConstr(E[p]   <= delta[p],                             name=f"MaxExp_p{p}")

m.addConstr(x25.sum() >= demand2025, name="Demand25")
m.addConstr(x35.sum() >= demand2035, name="Demand35")

## 3. Environmental Constraints

**Instructions:** Uncomment ONE block below for Parts b/c, or uncomment BOTH water and pollutant for Part d. Then re-run sections 3 and 4.

In [ ]:
# --- PART b/c: WATER constraint (uncomment to activate) ---
# water_constr = m.addConstr(
#     gp.quicksum(water[p] * x35[p] for p in processes) <= water_target,
#     name="WaterLimit35"
# )

# --- PART b: ELECTRICITY constraint (uncomment to activate) ---
# electricity_constr = m.addConstr(
#     gp.quicksum(electricity[p] * x35[p] for p in processes) <= electricity_target,
#     name="ElectricityLimit35"
# )

# --- PART b: FUEL constraint (uncomment to activate) ---
# fuel_constr = m.addConstr(
#     gp.quicksum(fuel[p] * x35[p] for p in processes) <= fuel_target,
#     name="FuelLimit35"
# )

# --- PART b/c: POLLUTANT constraint (uncomment to activate) ---
# pollutant_constr = m.addConstr(
#     gp.quicksum(pollutant[p] * x35[p] for p in processes) <= pollutant_target,
#     name="PollutantLimit35"
# )

## 4. Solve and Display Results

In [ ]:
m.optimize()

if m.status == GRB.OPTIMAL:
    print(f"{'='*55}")
    print(f"OPTIMAL TOTAL COST: ${m.objVal:,.2f}")
    print(f"(Base case for comparison: $203,000.00)")
    print(f"Cost increase vs base case: ${m.objVal - 203000:,.2f}")
    print(f"{'='*55}")

    print("\n--- 2025 Production Plan ---")
    for p in processes:
        print(f"  Process {p}: {x25[p].x:.2f} kt  (capacity = {cap0[p]} kt)")
        dual = cap25_constrs[p].Pi
        if abs(dual) > 1e-6:
            print(f"    → Shadow price: ${dual:.2f}/kt")

    print("\n--- 2035 Production & Expansion Plan ---")
    for p in processes:
        cap35 = cap0[p] + E[p].x
        print(f"  Process {p}: produce {x35[p].x:.2f} kt  "
              f"| capacity {cap35:.2f} kt  "
              f"| expanded by {E[p].x:.2f} kt")

    # Shadow prices for environmental constraints (Part c)
    print("\n--- Shadow Prices: Environmental Constraints (Part c) ---")
    for name, var in [("Water", "water_constr"),
                      ("Electricity", "electricity_constr"),
                      ("Fuel", "fuel_constr"),
                      ("Pollutant", "pollutant_constr")]:
        try:
            constr = eval(var)
            dual = constr.Pi
            print(f"  {name}: ${abs(dual):.2f} per unit reduction")
            print(f"    (Each additional unit reduction in {name.lower()} limit "
                  f"costs the company ${abs(dual):.2f} more)")
        except NameError:
            print(f"  {name}: Not active in this run")

elif m.status == GRB.INFEASIBLE:
    print("\nMODEL IS INFEASIBLE.")
    print("The environmental target cannot be met while satisfying demand.")
    print("This means no combination of processes can meet 200 kt demand")
    print("while staying within the environmental limit.")
else:
    print(f"No optimal solution found. Gurobi status code: {m.status}")

## 5. Summary of Results Across All Scenarios

| Scenario | Optimal Cost | vs Base Case | Feasible? |
|----------|-------------|--------------|----------|
| Base case (Part a) | $203,000 | — | ✅ |
| Water constraint only | $562,400 | +$359,400 | ✅ |
| Electricity constraint | — | — | ❌ Infeasible |
| Fuel constraint | — | — | ❌ Infeasible |
| Pollutant constraint only | $416,400 | +$213,400 | ✅ |
| Water + Pollutant (Part d) | $586,400 | +$383,400 | ✅ |

## 6. Key Findings

**Electricity and fuel constraints are infeasible** — no combination of processes can meet the 200 kt 2035 demand while hitting an 80% reduction target, given available expansion limits.

**Water constraint** ($2,050/unit shadow price) is the most expensive feasible policy — it forces the model away from cheap Process 1 (high water use) toward Process 3 (low water use but very high investment cost of $5,000/kt/yr).

**Pollutant constraint** ($512.50/unit shadow price) is less disruptive — Process 2 (low pollutant emitter at 2 units/kt) can absorb the extra demand without requiring Process 3.

**Combined constraint (Part d)** forces simultaneous use of Process 2 (low pollutant) and Process 3 (low water), pushing cost to $586,400 — higher than either constraint alone.